# Bilateral SDA drill-down dashboard

Interactive accounting-attribution dashboard for validated annual SDA reports. It preserves `emitter_country`, `emitter_sector`, `driver_country`, `driver_sector`, `consumer_country`, `demanded_sector`, `source_country`, `source_group`, and `source_sector` as distinct roles; signed negative contributions are never discarded. Set `SDA_RUN_ROOT` before starting Jupyter, or edit `RUN_ROOT` below.


In [ ]:
from pathlib import Path
from collections import defaultdict
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

RUN_ROOT = Path(os.environ.get('SDA_RUN_ROOT', '/local/s2053268/sda-campaign-2001-2014')).resolve()
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src' / 'sda').is_dir():
    REPO_ROOT = Path('..').resolve()
REGISTRY_PATH = REPO_ROOT / 'src' / 'sda' / 'output' / 'dashboard_reporting_contract_v2.json'
REGISTRY = json.loads(REGISTRY_PATH.read_text())
assert REGISTRY['language_policy'] == 'accounting_attribution'
assert len(REGISTRY['panels']) == 11
display(Markdown(f'**Campaign root:** `{RUN_ROOT}`  \n**Dashboard contract:** `{REGISTRY_PATH.name}` (v{REGISTRY["contract_version"]})'))


In [ ]:
def read_json(path):
    return json.loads(path.read_text())

annual_manifests = sorted((RUN_ROOT / 'annual').glob('*/manifest.v2.json'))
annual_inventory = []
for path in annual_manifests:
    manifest = read_json(path)
    annual_inventory.append({
        'run_key': path.parent.name, 'path': path.parent,
        'year_previous': manifest['year_pair'][0], 'year_current': manifest['year_pair'][1],
        'resolution': manifest['resolution'], 'units': manifest['units'],
        'validation_status': manifest['validation_status'],
        'hierarchy_rows': manifest['row_counts']['annual_hierarchy'],
        'drilldown_rows': manifest['row_counts']['annual_drilldown'],
    })
annual_runs = pd.DataFrame(annual_inventory)
panel_root = RUN_ROOT / 'panel' / 'full_panel' / 'cross_year_panel'
panel_paths = {spec['panel_id']: panel_root / f"{spec['panel_id']}.parquet" for spec in REGISTRY['panels']}
completion_path = RUN_ROOT / 'campaign_completion.json'
completion = read_json(completion_path) if completion_path.is_file() else {'status': 'campaign still running'}
display(Markdown('## Campaign inventory'))
display(pd.DataFrame([completion]).T.rename(columns={0: 'value'}))
display(annual_runs.drop(columns=['path']) if not annual_runs.empty else Markdown('*No accepted annual reports found yet.*'))
display(pd.DataFrame([{'panel_id': key, 'available': path.is_file(), 'path': str(path)} for key, path in panel_paths.items()]))


In [ ]:
NODE_LABELS = {
    'delta_g': 'Global emissions change', 'emission_intensity': 'Emission intensity',
    'leontief_production_structure': 'Leontief / production structure',
    'final_demand': 'Final demand', 'D_v': 'D_v', 'D_C': 'D_C',
    'D_A_x': 'D_A_x', 'D_T_coef': 'D_T_coef', 'D_w_coef': 'D_w_coef',
    'D_YF': 'D_YF', 'D_YB': 'D_YB', 'D_y': 'D_y',
}
METRICS = ['contribution', 'positive_contribution', 'negative_contribution', 'gross_contribution', 'net_pct_of_parent_gross']

def load_annual(run_key):
    root = RUN_ROOT / 'annual' / run_key
    hierarchy = pd.read_parquet(root / 'annual_hierarchy.parquet')
    drilldown = pd.read_parquet(root / 'annual_drilldown.parquet')
    manifest = read_json(root / 'manifest.v2.json')
    hierarchy = hierarchy.loc[hierarchy['is_publishable'] & hierarchy['validation_status'].eq('passed')].copy()
    drilldown = drilldown.loc[drilldown['is_publishable'] & drilldown['validation_status'].eq('passed')].copy()
    return hierarchy, drilldown, manifest

def signed_color(values):
    return ['#26734d' if value >= 0 else '#b33a3a' for value in values]

def ranked(frame, top_n):
    return frame.assign(abs_contribution=frame['contribution'].abs()).sort_values('abs_contribution', ascending=False).head(top_n)

def role_label(frame, grain):
    if grain not in frame or frame[grain].isna().all():
        return pd.Series(['(not applicable)'] * len(frame), index=frame.index)
    return frame[grain].fillna('(not classified)').astype(str)


In [ ]:
display(Markdown('## Annual hierarchy and drill-down explorer'))
if annual_runs.empty:
    display(Markdown('*Run the campaign reporting stage to enable the interactive explorer.*'))
else:
    run_widget = widgets.Dropdown(options=annual_runs['run_key'].tolist(), description='Transition:', layout=widgets.Layout(width='420px'))
    effect_widget = widgets.Dropdown(description='Effect:', layout=widgets.Layout(width='300px'))
    grain_widget = widgets.Dropdown(description='Grain:', layout=widgets.Layout(width='360px'))
    top_widget = widgets.IntSlider(value=20, min=5, max=50, step=5, description='Top |net|:')
    annual_output = widgets.Output()

    def set_effects(*_):
        _, drilldown, _ = load_annual(run_widget.value)
        effect_widget.options = sorted(drilldown['effect_id'].dropna().unique())

    def set_grains(*_):
        if effect_widget.value is None:
            return
        _, drilldown, _ = load_annual(run_widget.value)
        subset = drilldown.loc[drilldown['effect_id'].eq(effect_widget.value)]
        allowed = REGISTRY['effects'][effect_widget.value]['allowed_grains']
        present = set(subset['grain_id'].dropna().unique())
        grain_widget.options = [grain for grain in allowed if grain in present]

    def render_annual(*_):
        if not effect_widget.value or not grain_widget.value:
            return
        hierarchy, drilldown, manifest = load_annual(run_widget.value)
        effect, grain = effect_widget.value, grain_widget.value
        subset = drilldown.loc[drilldown['effect_id'].eq(effect) & drilldown['grain_id'].eq(grain)].copy()
        subset['label'] = role_label(subset, grain)
        subset = ranked(subset, top_widget.value).sort_values('contribution')
        with annual_output:
            clear_output(wait=True)
            display(Markdown(f'### {run_widget.value}: hierarchy'))
            shown = hierarchy[['node_id', 'parent_node_id', 'depth', *METRICS, 'denominator_value', 'denominator_status']].copy()
            shown['node'] = shown['node_id'].map(NODE_LABELS).fillna(shown['node_id'])
            display(shown[['node', 'parent_node_id', 'depth', *METRICS, 'denominator_value', 'denominator_status']])
            fig, axes = plt.subplots(1, 2, figsize=(16, max(5, 0.35 * max(len(subset), len(hierarchy)))))
            h = hierarchy.sort_values(['depth', 'node_id'])
            hlabels = [('  ' * int(depth)) + NODE_LABELS.get(node, node) for node, depth in zip(h['node_id'], h['depth'])]
            axes[0].barh(hlabels, h['contribution'], color=signed_color(h['contribution']))
            axes[0].axvline(0, color='black', linewidth=.8); axes[0].set_title('Signed hierarchy contribution')
            axes[1].barh(subset['label'], subset['contribution'], color=signed_color(subset['contribution']))
            axes[1].axvline(0, color='black', linewidth=.8); axes[1].set_title(f'{effect} by {grain} — top absolute net')
            for axis in axes: axis.set_xlabel(manifest['units'])
            plt.tight_layout(); plt.show()
            display(subset[['label', *METRICS, 'denominator_value', 'denominator_status']].sort_values('contribution', ascending=False))

    run_widget.observe(set_effects, names='value'); effect_widget.observe(set_grains, names='value')
    for control in (run_widget, effect_widget, grain_widget, top_widget): control.observe(render_annual, names='value')
    set_effects(); set_grains(); render_annual()
    display(widgets.VBox([widgets.HBox([run_widget, effect_widget]), widgets.HBox([grain_widget, top_widget]), annual_output]))


In [ ]:
display(Markdown('## Cross-year panel explorer'))
available_panels = [panel_id for panel_id, path in panel_paths.items() if path.is_file()]
if not available_panels:
    display(Markdown('*Build the panel stage to enable cross-year exploration.*'))
else:
    panel_widget = widgets.Dropdown(options=available_panels, description='Panel:', layout=widgets.Layout(width='350px'))
    resolution_widget = widgets.Dropdown(description='Resolution:', layout=widgets.Layout(width='280px'))
    panel_effect_widget = widgets.Dropdown(description='Effect:', layout=widgets.Layout(width='280px'))
    label_widget = widgets.SelectMultiple(description='Labels:', rows=8, layout=widgets.Layout(width='500px'))
    panel_output = widgets.Output()

    def load_panel():
        frame = pd.read_parquet(panel_paths[panel_widget.value])
        return frame.loc[frame['is_publishable'] & frame['validation_status'].eq('passed')].copy()

    def set_panel_options(*_):
        frame = load_panel()
        resolution_widget.options = sorted(frame['resolution'].dropna().unique())
        panel_effect_widget.options = sorted(frame['effect_id'].dropna().unique())

    def set_panel_labels(*_):
        if resolution_widget.value is None or panel_effect_widget.value is None:
            return
        frame = load_panel()
        frame = frame.loc[frame['resolution'].eq(resolution_widget.value) & frame['effect_id'].eq(panel_effect_widget.value)]
        totals = frame.groupby('label', dropna=False)['contribution'].apply(lambda values: values.abs().sum()).sort_values(ascending=False)
        options = [str(label) for label in totals.head(12).index]
        label_widget.options = options; label_widget.value = tuple(options[:min(5, len(options))])

    def render_panel(*_):
        if resolution_widget.value is None or panel_effect_widget.value is None:
            return
        frame = load_panel()
        frame = frame.loc[frame['resolution'].eq(resolution_widget.value) & frame['effect_id'].eq(panel_effect_widget.value)]
        frame['label'] = frame['label'].astype(str)
        if label_widget.value: frame = frame.loc[frame['label'].isin(label_widget.value)]
        with panel_output:
            clear_output(wait=True)
            if frame.empty:
                display(Markdown('*No rows for this selection.*')); return
            pivot = frame.pivot_table(index='year_current', columns='label', values='contribution', aggfunc='sum').sort_index()
            pivot.plot(marker='o', figsize=(14, 6)); plt.axhline(0, color='black', linewidth=.8)
            plt.title(f'{panel_widget.value}: {panel_effect_widget.value}'); plt.ylabel(frame['units'].iloc[0])
            plt.xlabel('Current year'); plt.tight_layout(); plt.show()
            display(frame.sort_values(['year_current', 'contribution'], ascending=[True, False])[[
                'year_previous', 'year_current', 'resolution', 'effect_id', 'grain_id', 'label', *METRICS, 'denominator_status']])

    panel_widget.observe(set_panel_options, names='value')
    resolution_widget.observe(set_panel_labels, names='value'); panel_effect_widget.observe(set_panel_labels, names='value')
    for control in (panel_widget, resolution_widget, panel_effect_widget, label_widget): control.observe(render_panel, names='value')
    set_panel_options(); set_panel_labels(); render_panel()
    display(widgets.VBox([widgets.HBox([panel_widget, resolution_widget, panel_effect_widget]), label_widget, panel_output]))


In [ ]:
display(Markdown('## Sourcing semantics and availability'))
source_products = [
    ('D_C', 'Intermediate-input source country', 'source_country', 'D_C.source_country_cube.parquet'),
    ('D_C', 'Intermediate-input source group × sector', 'source_group_sector', 'D_C.source_group_sector_cube.parquet'),
    ('D_YF', 'Final-demand source country', 'yf_source_country', 'D_YF.source_country_cube.parquet'),
    ('D_YF', 'Final-demand source group × sector', 'yf_source_group_sector', 'D_YF.source_group_sector_cube.parquet'),
]
source_rows = []
for run in ([] if annual_runs.empty else annual_runs['run_key']):
    for effect, meaning, directory, filename in source_products:
        path = RUN_ROOT / 'offshoring' / run / directory / filename
        source_rows.append({'run_key': run, 'effect_id': effect, 'meaning': meaning, 'available': path.is_file(), 'path': str(path)})
sources = pd.DataFrame(source_rows)
display(sources if not sources.empty else Markdown('*No sourcing inventory available.*'))
display(Markdown('D_C products retain structural-driver roles. D_YF products retain consumer country and demanded sector and never relabel either as a structural driver.'))


In [ ]:
display(Markdown('## Validation, denominators, and provenance'))
if annual_runs.empty:
    display(Markdown('*No annual manifests available.*'))
else:
    validation_rows, provenance_rows = [], []
    for item in annual_inventory:
        manifest = read_json(item['path'] / 'manifest.v2.json')
        for check, report in manifest['validation'].items():
            validation_rows.append({'run_key': item['run_key'], 'check': check, **report})
        for source in manifest.get('source_files', []):
            provenance_rows.append({'run_key': item['run_key'], **source})
    validation = pd.DataFrame(validation_rows)
    display(validation.sort_values(['run_key', 'check']))
    display(Markdown(f'**All mandatory report gates passed:** `{bool(validation["passed"].all())}`'))
    denominator_status = []
    for item in annual_inventory:
        hierarchy, drilldown, _ = load_annual(item['run_key'])
        for product, frame in [('annual_hierarchy', hierarchy), ('annual_drilldown', drilldown)]:
            counts = frame['denominator_status'].value_counts(dropna=False)
            denominator_status.extend({'run_key': item['run_key'], 'product': product, 'status': key, 'rows': value} for key, value in counts.items())
    display(pd.DataFrame(denominator_status))
    display(pd.DataFrame(provenance_rows) if provenance_rows else Markdown('*No source-file provenance records found.*'))
